In [ ]:
import os
import math
import numpy as np
import xarray as xr
import scipy.ndimage
from pathlib import Path
import netCDF4 as nc
import cc3d
import gc
import metpy
from metpy.calc import density
from metpy.units import units
import matplotlib.pyplot as plt
import scipy.linalg
import metpy.calc as mpcalc
import sys

#Constants
t = 6

# Input
input_dir = Path("/mnt/stor-pool-01/projects/heus/ShellAnalysis/full-area")

In [ ]:
#Check that files exist
path_ds_ql_mask = Path(input_dir / "ql_mask.nc")
if not path_ds_ql_mask.is_file():
    print(f"❌ ERROR: Simulation dataset not found at:\n   {path_ds_ql_mask}\nEnding program early.", file=sys.stderr)
    sys.exit(1)
path_ds_vpg = Path(input_dir / "vpg.nc")
if not path_ds_vpg.is_file():
    print(f"❌ ERROR: Simulation dataset not found at:\n   {path_ds_vpg}\nEnding program early.", file=sys.stderr)
    sys.exit(1)
path_ds_b = Path(input_dir / "b.nc")
if not path_ds_b.is_file():
    print(f"❌ ERROR: Simulation dataset not found at:\n   {path_ds_b}\nEnding program early.", file=sys.stderr)
    sys.exit(1)
path_ds_vpg_b = Path(input_dir / "vpg_b.nc")
if not path_ds_vpg_b.is_file():
    print(f"❌ ERROR: Simulation dataset not found at:\n   {path_ds_vpg_b}\nEnding program early.", file=sys.stderr)
    sys.exit(1)


ds_ql_mask = xr.open_dataset(path_ds_ql_mask, decode_times=False, chunks={'time': 1})
ds_vpg = xr.open_dataset(path_ds_vpg, decode_times=False, chunks={'time': 1})
ds_b = xr.open_dataset(path_ds_b, decode_times=False, chunks={'time': 1})
ds_vpg_b = xr.open_dataset(path_ds_vpg_b, decode_times=False, chunks={'time': 1})

In [5]:
#Getting zi (arbitrary z) at cloud base
ql_mask_slice = ds_ql_mask.isel(time=t)
has_nonzero_per_layer = (ql_mask_slice.ql_mask > 0).any(dim=["x", "y"])
smallest_z_idx = has_nonzero_per_layer.argmax(dim="z").compute().item()
arb_z =ql_mask_slice.z.isel(z=smallest_z_idx).compute().item()

In [ ]:
def compute_bounded_kinetic_energy(da, z_base, template_ds):
    """
    Computes cumulative kinetic energy from a specific lower bound 'z_base' 
    to the top of the domain, and realigns the output to match a template dataset.
    
    Parameters:
    -----------
    da : xr.DataArray
        The 3D spatial input field (e.g., buoyancy, VPG) for a single timestep.
    z_base : float
        The altitude value of the lower integration bound (e.g., cloud base).
    template_ds : xr.Dataset or xr.DataArray
        The full-grid template used to reindex the dimensions back to the complete domain.
    """
    # 1. Slice the variable from the lower bound to the top of the atmosphere
    da_bounded = da.sel(z=slice(z_base, None))
    
    # 2. Cumulative integration along the vertical axis
    ke_bounded = da_bounded.cumulative_integrate(dim="z")
    
    # 3. Shape back into the regular full-grid dimension, filling below z_base with 0.0
    ke_full = ke_bounded.reindex_like(template_ds, fill_value=0.0)
    
    return ke_full

In [ ]:
#Time slice vars
vpg_slice = ds_vpg.vpg.isel(time=t)
b_slice = ds_b.b.isel(time=t)
vpg_b_slice = ds_vpg_b.vpg_b.isel(time=t)
b_eff_slice = vpg_b_slice + b_slice

In [ ]:
#Convert to KE
reindex_template = ds_b.isel(time=t)

ke_vpg = compute_bounded_kinetic_energy(vpg_slice, arb_z, template_ds=reindex_template)
ke_b = compute_bounded_kinetic_energy(b_slice, arb_z, template_ds=reindex_template)
ke_vpg_b = compute_bounded_kinetic_energy(vpg_b_slice, arb_z, template_ds=reindex_template)
ke_b_eff = compute_bounded_kinetic_energy(b_eff_slice, arb_z, template_ds=reindex_template)